# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows `checklist.md`.

Data notes: there is no date column and every `device_serial` is unique (checked below), so there is no time ordering or repeated entity to respect; a stratified random split is the appropriate split. `device_serial` is an identifier and is dropped. The remaining columns (channel, device age, price, daily hours) are known at sale/registration, i.e. at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED = 0
df = pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', int(df.isna().sum().sum()))
print('claim rate:', df.claim_within_year.mean())
df.describe()

## 1. Hold out the test set before any modelling
20% test, stratified. Entities are unique, so the split is trivially by whole entity.

In [ ]:
y = df.claim_within_year
X = df.drop(columns=['claim_within_year', 'device_serial'])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(len(X_tr), len(X_te), y_tr.mean(), y_te.mean())

## 2. Candidate models (named before scoring)
1. **Baseline**: predicts the training prevalence.
2. **Logistic regression** (one-hot channel, scaled numerics), grid over `C`.
3. **Gradient boosting** (HistGradientBoosting), small grid over learning rate / depth / leaf size.

Selection metric: CV ROC AUC (positives are ~12.6%, so accuracy is not used). The decision threshold is chosen on out-of-fold predictions to maximise F1. All preprocessing sits inside the Pipeline, so it is refit within each CV fold.

In [ ]:
cat = ['channel']; num = ['device_age_years', 'price_eur', 'daily_hours']
def pre(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
cands = {
    'baseline': (Pipeline([('p', pre(False)), ('m', DummyClassifier(strategy='prior'))]), {}),
    'logreg': (Pipeline([('p', pre(True)), ('m', LogisticRegression(max_iter=2000))]), {'m__C': [0.01, 0.1, 1, 10]}),
    'hgb': (Pipeline([('p', pre(False)), ('m', HistGradientBoostingClassifier(random_state=SEED))]),
            {'m__learning_rate': [0.03, 0.1], 'm__max_depth': [2, 3, None], 'm__min_samples_leaf': [20, 100]}),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

## 3. Cross-validation on training rows only

In [ ]:
res = {}
for n, (p, g) in cands.items():
    gs = GridSearchCV(p, g or {'m__strategy': ['prior']}, scoring='roc_auc', cv=cv).fit(X_tr, y_tr)
    res[n] = gs
    print(f"{n:9s} CV AUC={gs.best_score_:.4f} params={gs.best_params_ if g else ''}")

In [ ]:
# threshold from out-of-fold probabilities of the best CV model (training rows only)
best = max(res, key=lambda n: res[n].best_score_)
oof = cross_val_predict(res[best].best_estimator_, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
ths = np.linspace(0.02, 0.6, 59)
f1s = [f1_score(y_tr, oof >= t) for t in ths]
thr = ths[int(np.argmax(f1s))]
print(best, 'OOF AP=%.4f' % average_precision_score(y_tr, oof), 'prevalence=%.3f' % y_tr.mean(),
      'threshold=%.2f' % thr, 'OOF F1=%.3f' % max(f1s))

## 4. Choice (fixed before touching the test set)
The model, hyperparameters and threshold are those printed above: best CV AUC, and the threshold maximising out-of-fold F1. Nothing changes after this point.

## 5. Refit on all training rows; score the test set once

In [ ]:
final = res[best].best_estimator_.fit(X_tr, y_tr)
p = final.predict_proba(X_te)[:, 1]
pred = p >= thr
base = res['baseline'].best_estimator_.predict_proba(X_te)[:, 1]
print('Chosen model:', best)
print('Test ROC AUC : %.4f (baseline %.4f)' % (roc_auc_score(y_te, p), roc_auc_score(y_te, base)))
print('Test PR AUC  : %.4f (prevalence %.4f)' % (average_precision_score(y_te, p), y_te.mean()))
print('At threshold %.2f: precision %.3f recall %.3f F1 %.3f' % (thr, precision_score(y_te, pred), recall_score(y_te, pred), f1_score(y_te, pred)))

## Interpretation
The printed test numbers are the headline. The test set was scored once and no tuning followed it.